In [6]:
import pandas as pd
import heapq

In [10]:
df = pd.read_csv("data/processed/edges_clean.csv")

graph = {}
for _, row in df.iterrows():
    graph.setdefault(int(row["from_node"]), {})[
        int(row["to_node"])
    ] = int(row["cost"])

print("Graph loaded successfully")

Graph loaded successfully


In [11]:
import os
print(os.getcwd())

D:\DeliveryRouteOptimizer_DAA


In [12]:
def dijkstra_with_path(graph, start):
    dist = {node: float("inf") for node in graph}
    prev = {node: None for node in graph}

    dist[start] = 0
    pq = [(0, start)]

    while pq:
        current_dist, current_node = heapq.heappop(pq)

        if current_dist > dist[current_node]:
            continue

        for neighbor, cost in graph.get(current_node, {}).items():
            new_dist = current_dist + cost

            if new_dist < dist[neighbor]:
                dist[neighbor] = new_dist
                prev[neighbor] = current_node
                heapq.heappush(pq, (new_dist, neighbor))

    return dist, prev

In [13]:
def heuristic(node, goal):
    return abs(node - goal)

def astar(graph, start, goal):
    open_set = [(0, start)]
    g_cost = {start: 0}
    prev = {start: None}

    while open_set:
        _, current = heapq.heappop(open_set)

        if current == goal:
            break

        for neighbor, cost in graph.get(current, {}).items():
            tentative = g_cost[current] + cost

            if neighbor not in g_cost or tentative < g_cost[neighbor]:
                g_cost[neighbor] = tentative
                f_cost = tentative + heuristic(neighbor, goal)
                heapq.heappush(open_set, (f_cost, neighbor))
                prev[neighbor] = current

    return g_cost, prev

In [14]:
def reconstruct_path(prev, start, target):
    path = []
    current = target

    while current is not None:
        path.append(current)
        current = prev.get(current)

    path.reverse()

    return path if path and path[0] == start else []

In [15]:
def shortest_path_dijkstra(graph, start, end):
    dist, prev = dijkstra_with_path(graph, start)
    path = reconstruct_path(prev, start, end)
    return path, dist[end]

def shortest_path_astar(graph, start, end):
    costs, prev = astar(graph, start, end)
    path = reconstruct_path(prev, start, end)
    return path, costs[end]

In [16]:
def build_distance_matrix(graph, nodes, method="dijkstra"):
    matrix = {}

    for i in nodes:
        matrix[i] = {}
        for j in nodes:
            if i != j:
                if method == "dijkstra":
                    _, cost = shortest_path_dijkstra(graph, i, j)
                else:
                    _, cost = shortest_path_astar(graph, i, j)

                matrix[i][j] = cost

    return matrix

In [17]:
def tsp_nearest_neighbor(distance_matrix, start, deliveries):
    unvisited = set(deliveries)
    route = [start]
    total_cost = 0
    current = start

    while unvisited:
        next_node = min(unvisited, key=lambda node: distance_matrix[current][node])
        total_cost += distance_matrix[current][next_node]
        route.append(next_node)
        current = next_node
        unvisited.remove(next_node)

    return route, total_cost

In [18]:
def build_full_path(graph, route, method="dijkstra"):
    full_path = []

    for i in range(len(route) - 1):
        if method == "dijkstra":
            path, _ = shortest_path_dijkstra(graph, route[i], route[i+1])
        else:
            path, _ = shortest_path_astar(graph, route[i], route[i+1])

        if i == 0:
            full_path.extend(path)
        else:
            full_path.extend(path[1:])

    return full_path

In [19]:
start_node = 1
delivery_nodes = [3, 5, 7]

all_nodes = [start_node] + delivery_nodes

# Build distance matrices
dist_matrix_dij = build_distance_matrix(graph, all_nodes, "dijkstra")
dist_matrix_astar = build_distance_matrix(graph, all_nodes, "astar")

# Solve TSP
route_dij, cost_dij = tsp_nearest_neighbor(
    dist_matrix_dij, start_node, delivery_nodes
)

route_astar, cost_astar = tsp_nearest_neighbor(
    dist_matrix_astar, start_node, delivery_nodes
)

# Expand full route
final_path_dij = build_full_path(graph, route_dij, "dijkstra")
final_path_astar = build_full_path(graph, route_astar, "astar")

print("Dijkstra Multi-Stop Order:", route_dij)
print("Dijkstra Total Cost:", cost_dij)
print("Dijkstra Full Path:", final_path_dij)

print("\nA* Multi-Stop Order:", route_astar)
print("A* Total Cost:", cost_astar)
print("A* Full Path:", final_path_astar)

Dijkstra Multi-Stop Order: [1, 3, 5, 7]
Dijkstra Total Cost: 32
Dijkstra Full Path: [1, 2, 3, 5, 4, 7]

A* Multi-Stop Order: [1, 3, 5, 7]
A* Total Cost: 32
A* Full Path: [1, 2, 3, 5, 4, 7]
